# Build ShopKart's first data pipeline (solution)

**Video 7.2** | ShopKart project

**Goal:** Turn the Module 6 cleaning into an automatic pipeline: extract() → transform() → quality_checks() → load() into a SQLite warehouse with a star schema, then add January 2026 orders with an incremental load.

### How to open in Colab
1. Go to colab.research.google.com → **File → Upload notebook** → choose this `.ipynb` file.
2. Click the **folder icon** on the left sidebar (Files panel) and drag in the data files: `orders_raw.csv`, `customers.csv`, `products.csv`, `new_orders_2026_01.csv`.
3. Run cells one by one with **Shift + Enter**.

> Files you upload to the Files panel disappear when the runtime resets. If you see `FileNotFoundError`, upload them again.

*Optional:* to check that this pipeline makes exactly the same file as Module 6, also upload your Module 6 `shopkart_clean.csv` after renaming it to `module6_shopkart_clean.csv` (step 4 compares the two).

## The plan

```
source files ──► extract() ──► transform() ──► quality_checks() ──► load() ──► shopkart_warehouse.db
                                                    │ fail                     └──► shopkart_clean.csv
                                                    ▼
                                             STOP, load nothing
```

In Module 6 we cleaned the data by running cells by hand. A **pipeline** puts the same steps into **functions**, so one command runs everything, in the same order, every time. `log()` prints what happened with the time, like a diary of the run.

In [1]:
import sqlite3
from datetime import datetime

import pandas as pd

# Settings: all file names in one place
ORDERS_FILE = "orders_raw.csv"
CUSTOMERS_FILE = "customers.csv"
PRODUCTS_FILE = "products.csv"
NEW_ORDERS_FILE = "new_orders_2026_01.csv"
WAREHOUSE_DB = "shopkart_warehouse.db"
CLEAN_CSV = "shopkart_clean.csv"
LOG_FILE = "pipeline_log.txt"

run_log = []   # every log() message is also stored here

def log(message):
    """Print a message with the time, and remember it for the run log."""
    line = datetime.now().strftime("%H:%M:%S") + "  " + message
    print(line)
    run_log.append(line)

log("PIPELINE START")

05:22:29  PIPELINE START


## Step 1 – Extract: read the sources as they are
Extract never changes the data. It only reads it and tells us how much came in.

In [2]:
def extract(orders_file=ORDERS_FILE):
    orders = pd.read_csv(orders_file)
    customers = pd.read_csv(CUSTOMERS_FILE)
    products = pd.read_csv(PRODUCTS_FILE)
    log(f"EXTRACT  {orders_file}: {len(orders)} rows | customers: {len(customers)} | products: {len(products)}")
    return orders, customers, products

orders, customers, products = extract()

05:22:29  EXTRACT  orders_raw.csv: 5055 rows | customers: 800 | products: 60


## Step 2 – Transform: the Module 6 cleaning inside one function
This is the same code as the Module 6 pipeline notebook, copied into a function. Notice step 4: we read the `YYYY-MM-DD` dates first and only then the `DD-MM-YYYY` ones, so no date gets its day and month swapped.

In [3]:
def transform(orders, customers, products):
    df = orders.copy()

    # 1. Remove exact duplicate rows
    df = df.drop_duplicates()

    # 2. One spelling per city
    df["city"] = df["city"].astype(str).str.strip().str.title()
    df["city"] = df["city"].replace({"New Delhi": "Delhi", "Bangalore": "Bengaluru", "Bombay": "Mumbai",
                                     "Madras": "Chennai", "Calcutta": "Kolkata"})

    # 3. Prices like "₹1,299" -> 1299
    df["unit_price"] = (df["unit_price"].astype(str)
                        .str.replace("₹", "", regex=False)
                        .str.replace(",", "", regex=False)
                        .astype(float).astype(int))

    # 4. Dates: read YYYY-MM-DD first, then fill the gaps with DD-MM-YYYY
    #    (never format="mixed", dayfirst=True: it flips ISO dates like 2025-01-02)
    text_dates = df["order_date"].astype(str)
    df["order_date"] = pd.to_datetime(text_dates, format="%Y-%m-%d", errors="coerce").fillna(
                       pd.to_datetime(text_dates, format="%d-%m-%Y", errors="coerce"))

    # 5. Payment mode: one spelling for UPI, blanks -> "Unknown"
    df["payment_mode"] = df["payment_mode"].replace({"upi": "UPI"}).fillna("Unknown")

    # 6. Blank discount means no discount
    df["discount"] = df["discount"].astype(float).fillna(0.0)

    # 7. Returned: Yes/No -> Y/N
    df["returned"] = df["returned"].replace({"Yes": "Y", "No": "N"})

    # 8. Quantity 99 is a typo for 1 (real bulk orders of 10 are kept)
    df.loc[df["quantity"] == 99, "quantity"] = 1

    # 9. rating stays blank where the customer did not rate

    # Merge product and customer details (LEFT joins keep every order)
    df = df.merge(products[["product_id", "product_name", "category", "cost_price"]], on="product_id", how="left")
    df = df.merge(customers[["customer_id", "gender", "age", "membership"]], on="customer_id", how="left")

    # New columns
    df["revenue"] = (df["quantity"] * df["unit_price"] * (1 - df["discount"])).round(2)
    df["profit"] = df["revenue"] - df["quantity"] * df["cost_price"]
    df["order_month"] = df["order_date"].dt.month

    log(f"TRANSFORM  {len(orders)} raw rows -> {len(df)} clean rows, {df.shape[1]} columns")
    return df

clean = transform(orders, customers, products)
clean.head(3)

05:22:29  TRANSFORM  5055 raw rows -> 5000 clean rows, 21 columns


,order_id,order_date,customer_id,product_id,quantity,unit_price,discount,city,payment_mode,delivery_days,...,returned,product_name,category,cost_price,gender,age,membership,revenue,profit,order_month
0,O10001,2025-01-01,C0593,P048,6,362,0.20,Delhi,COD,4,...,N,Wings of Fire,Books,240,Female,26,Prime,1737.6,297.6,1
1,O10002,2025-01-01,C0367,P049,2,300,0.05,Chennai,Debit Card,4,...,N,Face Wash,Beauty,150,Male,35,Free,570.0,270.0,1
2,O10003,2025-01-01,C0707,P020,1,2258,0.05,Delhi,Debit Card,2,...,N,Saree,Fashion,1280,Male,41,Prime,2145.1,865.1,1


## Step 3 – Quality checks: stop if something looks wrong
A pipeline runs when nobody is watching (for example at 2 am). So it must check its own work. If any check fails, we `raise` an error: the pipeline stops and **nothing is loaded**. Bad data never reaches the people who use the warehouse.

In [4]:
def quality_checks(df, expected_rows):
    checks = {
        "row count = unique orders in source": len(df) == expected_rows,
        "no duplicate order_id": df["order_id"].is_unique,
        "no missing keys (order_id, customer_id, product_id, order_date)":
            df[["order_id", "customer_id", "product_id", "order_date"]].notnull().all().all(),
        "every product found in products.csv": df["category"].notnull().all(),
        "every customer found in customers.csv": df["membership"].notnull().all(),
        "unit_price > 0": (df["unit_price"] > 0).all(),
        "quantity between 1 and 10": df["quantity"].between(1, 10).all(),
    }
    failed = [name for name, ok in checks.items() if not ok]
    for name, ok in checks.items():
        print("   ", "PASS" if ok else "FAIL", "-", name)
    if failed:
        log(f"QUALITY CHECKS FAILED: {failed}")
        raise ValueError("Data quality check failed: " + "; ".join(failed) + ". Nothing was loaded.")
    log(f"QUALITY CHECKS  all {len(checks)} passed")

quality_checks(clean, expected_rows=orders["order_id"].nunique())

    PASS - row count = unique orders in source
    PASS - no duplicate order_id
    PASS - no missing keys (order_id, customer_id, product_id, order_date)
    PASS - every product found in products.csv
    PASS - every customer found in customers.csv
    PASS - unit_price > 0
    PASS - quantity between 1 and 10
05:22:29  QUALITY CHECKS  all 7 passed


### What a failure looks like
We break a **copy** of the clean data on purpose: we add two orders twice, set one price to 0 and delete one customer_id. The checks should catch all of it. (`try / except` lets us print the error and keep the notebook running. Without it you would see a red `ValueError`.)

In [5]:
broken = pd.concat([clean, clean.head(2)]).reset_index(drop=True)   # 2 orders now appear twice
broken.loc[5, "unit_price"] = 0                                       # a price of 0
broken.loc[7, "customer_id"] = None                                   # a missing key

try:
    quality_checks(broken, expected_rows=5000)
    print("Loaded")      # we never reach this line
except ValueError as error:
    print("PIPELINE STOPPED ->", error)

    FAIL - row count = unique orders in source
    FAIL - no duplicate order_id
    FAIL - no missing keys (order_id, customer_id, product_id, order_date)
    PASS - every product found in products.csv
    PASS - every customer found in customers.csv
    FAIL - unit_price > 0
    PASS - quantity between 1 and 10
05:22:29  QUALITY CHECKS FAILED: ['row count = unique orders in source', 'no duplicate order_id', 'no missing keys (order_id, customer_id, product_id, order_date)', 'unit_price > 0']
PIPELINE STOPPED -> Data quality check failed: row count = unique orders in source; no duplicate order_id; no missing keys (order_id, customer_id, product_id, order_date); unit_price > 0. Nothing was loaded.


## Step 4 – Load: a star schema in a SQLite warehouse

| Table | One row per | Columns |
|---|---|---|
| **fact_orders** (centre) | order | keys (order_id, date_key, customer_id, product_id) + numbers (quantity, unit_price, discount, revenue, profit …) |
| dim_customer | customer | name, gender, age, city, signup_date, membership |
| dim_product | product | product_name, category, list_price, cost_price |
| dim_date | date | date, year, month, month_name, quarter, weekday, is_weekend |

The fact table holds the **events and numbers**. The dimension tables **describe** them. `date_key` is the date written as a number: 2025-10-31 → 20251031.

In [6]:
def make_dim_date(df):
    """One row per order date, with handy calendar columns."""
    dates = pd.Series(df["order_date"].drop_duplicates().sort_values())
    dim_date = pd.DataFrame({
        "date_key": dates.dt.strftime("%Y%m%d").astype(int),   # 2025-01-31 -> 20250131
        "date": dates.dt.strftime("%Y-%m-%d"),
        "year": dates.dt.year,
        "month": dates.dt.month,
        "month_name": dates.dt.month_name(),
        "quarter": dates.dt.quarter,
        "weekday": dates.dt.day_name(),
        "is_weekend": dates.dt.dayofweek >= 5,
    })
    return dim_date.reset_index(drop=True)


def make_fact_orders(df):
    """The fact table: one row per order, keys + numbers only (no product names, no customer names)."""
    fact = df[["order_id", "customer_id", "product_id", "quantity", "unit_price", "discount",
               "city", "payment_mode", "delivery_days", "rating", "returned", "revenue", "profit"]].copy()
    fact.insert(1, "date_key", df["order_date"].dt.strftime("%Y%m%d").astype(int))
    return fact


make_fact_orders(clean).head(3)

,order_id,date_key,customer_id,product_id,quantity,unit_price,discount,city,payment_mode,delivery_days,rating,returned,revenue,profit
0,O10001,20250101,C0593,P048,6,362,0.20,Delhi,COD,4,2.0,N,1737.6,297.6
1,O10002,20250101,C0367,P049,2,300,0.05,Chennai,Debit Card,4,5.0,N,570.0,270.0
2,O10003,20250101,C0707,P020,1,2258,0.05,Delhi,Debit Card,2,3.0,N,2145.1,865.1


In [7]:
def load(df, customers, products):
    fact_orders = make_fact_orders(df)
    dim_customer = customers.copy()      # all 800 customers, even those who never ordered
    dim_product = products.copy()        # all 60 products
    dim_date = make_dim_date(df)

    conn = sqlite3.connect(WAREHOUSE_DB)
    fact_orders.to_sql("fact_orders", conn, if_exists="replace", index=False)
    dim_customer.to_sql("dim_customer", conn, if_exists="replace", index=False)
    dim_product.to_sql("dim_product", conn, if_exists="replace", index=False)
    dim_date.to_sql("dim_date", conn, if_exists="replace", index=False)
    conn.close()

    df.to_csv(CLEAN_CSV, index=False)
    log(f"LOAD  {WAREHOUSE_DB}: fact_orders {len(fact_orders)}, dim_customer {len(dim_customer)}, "
        f"dim_product {len(dim_product)}, dim_date {len(dim_date)} rows | {CLEAN_CSV} {df.shape}")


def table_sizes():
    """Count the rows in every warehouse table."""
    conn = sqlite3.connect(WAREHOUSE_DB)
    sizes = {t: int(pd.read_sql(f"SELECT COUNT(*) AS n FROM {t}", conn)["n"][0])
             for t in ["fact_orders", "dim_customer", "dim_product", "dim_date"]}
    conn.close()
    return sizes


load(clean, customers, products)
table_sizes()

05:22:29  LOAD  shopkart_warehouse.db: fact_orders 5000, dim_customer 800, dim_product 60, dim_date 336 rows | shopkart_clean.csv (5000, 21)


{'fact_orders': 5000, 'dim_customer': 800, 'dim_product': 60, 'dim_date': 336}

### Did the pipeline reproduce Module 6?
Same shape and same total revenue as Module 6. If you uploaded your Module 6 file as `module6_shopkart_clean.csv`, we also compare every value.

In [8]:
import os

check = pd.read_csv(CLEAN_CSV)
print("shopkart_clean.csv:", check.shape)                     # Module 6: (5000, 21)
print("Total revenue: ₹", round(check["revenue"].sum(), 2))   # Module 6: 14704649.7

if os.path.exists("module6_shopkart_clean.csv"):
    pd.testing.assert_frame_equal(check, pd.read_csv("module6_shopkart_clean.csv"))   # error if anything differs
    print("Identical to the Module 6 file: same shape, columns and values")
else:
    print("(Optional exact check skipped: module6_shopkart_clean.csv not uploaded)")

shopkart_clean.csv: (5000, 21)
Total revenue: ₹ 14704649.7
Identical to the Module 6 file: same shape, columns and values


## Step 5 – Incremental load: January 2026 orders arrive
Every month the shop sends a new file. Reloading 5,000 old orders every time is slow, and blindly appending can create duplicates. Let's look at the new file first.

In [9]:
new = pd.read_csv(NEW_ORDERS_FILE)
print("Rows in file:", len(new))
print("Exact duplicate rows:", new.duplicated().sum())
print("order_ids already loaded in 2025:", sorted(set(new["order_id"]) & set(clean["order_id"])))
new.head(6)

Rows in file: 40
Exact duplicate rows: 2
order_ids already loaded in 2025: ['O14999', 'O15000']


,order_id,order_date,customer_id,product_id,quantity,unit_price,discount,city,payment_mode,delivery_days,rating,returned
0,O14999,2025-12-28,C0579,P043,1,596,0.15,Mumbai,UPI,2,4.0,N
1,O15000,2025-12-28,C0326,P002,2,2499,0.10,Lucknow,Debit Card,7,2.0,Y
2,O15001,2026-01-01,C0352,P051,3,254,0.20,Mumbai,UPI,8,5.0,N
3,O15002,2026-01-03,C0164,P010,2,19423,0.05,Hyderabad,UPI,4,5.0,N
4,O15003,2026-01-04,C0134,P039,1,407,0.20,delhi,UPI,6,4.0,Y
5,O15004,04-01-2026,C0038,P009,3,15549,0.30,Bengaluru,COD,6,4.0,Y


The file has the same mess as before (city spellings, a `₹` price, blank discounts, two date styles) – our `transform()` already handles all of it. The new idea is the filter: **keep only order_ids the warehouse has not seen**, then `append` them.

In [10]:
def incremental_load(new_file):
    log(f"INCREMENTAL LOAD  starting with {new_file}")
    new_raw, customers, products = extract(new_file)
    new_df = transform(new_raw, customers, products)
    quality_checks(new_df, expected_rows=new_raw["order_id"].nunique())

    conn = sqlite3.connect(WAREHOUSE_DB)
    loaded_ids = set(pd.read_sql("SELECT order_id FROM fact_orders", conn)["order_id"])
    to_add = new_df[~new_df["order_id"].isin(loaded_ids)]          # keep only order_ids we have not loaded
    skipped = len(new_df) - len(to_add)

    make_fact_orders(to_add).to_sql("fact_orders", conn, if_exists="append", index=False)

    # add calendar rows for dates the warehouse does not have yet
    known_dates = set(pd.read_sql("SELECT date_key FROM dim_date", conn)["date_key"])
    new_dates = make_dim_date(to_add)
    new_dates = new_dates[~new_dates["date_key"].isin(known_dates)]
    new_dates.to_sql("dim_date", conn, if_exists="append", index=False)
    conn.close()

    log(f"INCREMENTAL LOAD  {len(new_raw)} rows in file, {len(new_raw) - len(new_df)} duplicate rows removed | "
        f"added {len(to_add)} new orders | skipped {skipped} already in warehouse | "
        f"added {len(new_dates)} new dates")
    return len(to_add), skipped

incremental_load(NEW_ORDERS_FILE)
table_sizes()

05:22:29  INCREMENTAL LOAD  starting with new_orders_2026_01.csv
05:22:29  EXTRACT  new_orders_2026_01.csv: 40 rows | customers: 800 | products: 60
05:22:29  TRANSFORM  40 raw rows -> 38 clean rows, 21 columns
    PASS - row count = unique orders in source
    PASS - no duplicate order_id
    PASS - no missing keys (order_id, customer_id, product_id, order_date)
    PASS - every product found in products.csv
    PASS - every customer found in customers.csv
    PASS - unit_price > 0
    PASS - quantity between 1 and 10
05:22:29  QUALITY CHECKS  all 7 passed
05:22:30  INCREMENTAL LOAD  40 rows in file, 2 duplicate rows removed | added 36 new orders | skipped 2 already in warehouse | added 21 new dates


{'fact_orders': 5036, 'dim_customer': 800, 'dim_product': 60, 'dim_date': 357}

### Run it again: nothing should change
A good pipeline is **safe to re-run**. If the 2 am job runs twice by mistake, we must not get double orders.

In [11]:
incremental_load(NEW_ORDERS_FILE)
table_sizes()

05:22:30  INCREMENTAL LOAD  starting with new_orders_2026_01.csv
05:22:30  EXTRACT  new_orders_2026_01.csv: 40 rows | customers: 800 | products: 60
05:22:30  TRANSFORM  40 raw rows -> 38 clean rows, 21 columns
    PASS - row count = unique orders in source
    PASS - no duplicate order_id
    PASS - no missing keys (order_id, customer_id, product_id, order_date)
    PASS - every product found in products.csv
    PASS - every customer found in customers.csv
    PASS - unit_price > 0
    PASS - quantity between 1 and 10
05:22:30  QUALITY CHECKS  all 7 passed
05:22:30  INCREMENTAL LOAD  40 rows in file, 2 duplicate rows removed | added 0 new orders | skipped 38 already in warehouse | added 0 new dates


{'fact_orders': 5036, 'dim_customer': 800, 'dim_product': 60, 'dim_date': 357}

## Step 6 – Ask the warehouse a question with SQL
This is a Module 4 style query: the fact table in the middle, JOINed to two dimension tables. `pd.read_sql` runs it and gives back a DataFrame.

In [12]:
REVENUE_QUERY = """
SELECT d.year, d.month, p.category,
       COUNT(*)                 AS orders,
       ROUND(SUM(f.revenue), 2) AS revenue
FROM fact_orders AS f
JOIN dim_product AS p ON f.product_id = p.product_id
JOIN dim_date    AS d ON f.date_key   = d.date_key
GROUP BY d.year, d.month, p.category
ORDER BY d.year, d.month, revenue DESC
"""

conn = sqlite3.connect(WAREHOUSE_DB)
result = pd.read_sql(REVENUE_QUERY, conn)
conn.close()
result.tail(10)

,year,month,category,orders,revenue
55,2025,12,Electronics,120,785434.50
56,2025,12,Fashion,160,395501.45
57,2025,12,Home,93,218227.85
58,2025,12,Beauty,83,96363.00
59,2025,12,Books,70,63375.25
60,2026,1,Electronics,8,107268.65
61,2026,1,Home,4,33181.60
62,2026,1,Fashion,7,20123.60
63,2026,1,Books,10,6363.90
64,2026,1,Beauty,7,6211.40


In [13]:
# Total per year: 2025 should match Module 6 exactly
result.groupby("year")[["orders", "revenue"]].sum()

,orders,revenue
year,,
2025,5000,14704649.70
2026,36,173149.15


## Step 7 – The run log
Every step wrote one line. In a real company this file is the first thing an engineer reads when something breaks.

In [14]:
log("PIPELINE END  success")
print("\n".join(run_log))

with open(LOG_FILE, "w", encoding="utf-8") as f:
    f.write("\n".join(run_log) + "\n")

05:22:30  PIPELINE END  success
05:22:29  PIPELINE START
05:22:29  EXTRACT  orders_raw.csv: 5055 rows | customers: 800 | products: 60
05:22:29  TRANSFORM  5055 raw rows -> 5000 clean rows, 21 columns
05:22:29  QUALITY CHECKS  all 7 passed
05:22:29  QUALITY CHECKS FAILED: ['row count = unique orders in source', 'no duplicate order_id', 'no missing keys (order_id, customer_id, product_id, order_date)', 'unit_price > 0']
05:22:29  LOAD  shopkart_warehouse.db: fact_orders 5000, dim_customer 800, dim_product 60, dim_date 336 rows | shopkart_clean.csv (5000, 21)
05:22:29  INCREMENTAL LOAD  starting with new_orders_2026_01.csv
05:22:29  EXTRACT  new_orders_2026_01.csv: 40 rows | customers: 800 | products: 60
05:22:29  TRANSFORM  40 raw rows -> 38 clean rows, 21 columns
05:22:29  QUALITY CHECKS  all 7 passed
05:22:30  INCREMENTAL LOAD  40 rows in file, 2 duplicate rows removed | added 36 new orders | skipped 2 already in warehouse | added 21 new dates
05:22:30  INCREMENTAL LOAD  starting with 

## Wrap-up
- One command now does what took a whole module by hand: **extract → transform → check → load**.
- The same code is in **`shopkart_pipeline.py`**. Upload it with the four data files and run it in a Colab cell with `!python shopkart_pipeline.py`. It rebuilds the warehouse, loads January, runs the query and saves `pipeline_log.txt`.
- Download `shopkart_warehouse.db` and `shopkart_clean.csv` from the Files panel (right-click → Download). **Module 8 (Visualization)** starts from `shopkart_clean.csv`.

**Practice:** add one more quality check (discount must be between 0 and 0.5) and test it on a broken copy.